# Лабораторная работа № 4.2
## Исследование нелинейной зависимости

**Выполнили:**

- Левченков Артемий Викторович;
- Бикбулатова Динара Фануровна;
- Ладыгин Максим Александрович.

**Группа:** БИВТ-24-АЭ-3

**Исходные данные:** дискретный и интервальный ряды возраста людей, совершивших преступления в Москве в 2021 году, из ЛР 1.

**Цель:** вычислить внутригрупповую и межгрупповую дисперсии, проверить формулу общей дисперсии, найти корреляционное отношение и коэффициент Спирмена.

### Исходные данные и интервальный ряд

Число интервалов $k=7$, ширина $h=\lceil(x_{\max}-x_{\min})/k\rceil$.
Левая граница включается, правая не включается; у последнего интервала включаются обе границы.


In [1]:
import pandas as pd
from math import ceil

data = pd.read_table("data/moscow.txt", header=None)[0]
N = len(data)

discrete = data.value_counts().sort_index().rename_axis("x").reset_index(name="n")
k = 7
x_min = data.min()
x_max = data.max()
h = ceil((x_max - x_min) / k)
edges = [x_min + i * h for i in range(k + 1)]

rows = []
for i in range(k):
    left = edges[i]
    right = edges[i + 1]
    if i < k - 1:
        n = ((data >= left) & (data < right)).sum()
    else:
        n = ((data >= left) & (data <= right)).sum()
    rows.append([left, right, (left + right) / 2, n])

interval = pd.DataFrame(rows, columns=["x_l", "x_r", "x_middle", "n"])
interval["w"] = interval["n"] / N
interval["S"] = interval["n"].cumsum()

print("Количество людей N =", N)
print("Количество различных возрастов =", len(discrete))
print("Ширина интервала h =", h)
display(interval.round(4))

Количество людей N = 32423
Количество различных возрастов = 58
Ширина интервала h = 9


,x_l,x_r,x_middle,n,w,S
0,14,23,18.5,4811,0.1484,4811
1,23,32,27.5,9476,0.2923,14287
2,32,41,36.5,7243,0.2234,21530
3,41,50,45.5,7951,0.2452,29481
4,50,59,54.5,1140,0.0352,30621
5,59,68,63.5,1472,0.0454,32093
6,68,77,72.5,330,0.0102,32423


### 1. Внутригрупповая и межгрупповая дисперсии

#### Групповые средние и дисперсии

Обозначим через $I_j$ множество возрастов в группе $j$, через $n_i$ — частоту возраста $x_i$.

$$N_j=\sum_{i\in I_j}n_i,\qquad
\bar x_j=\frac{\sum_{i\in I_j}x_i n_i}{N_j},\qquad
D_j=\frac{\sum_{i\in I_j}(x_i-\bar x_j)^2 n_i}{N_j}.$$

$$\bar x=\frac{\sum_i x_i n_i}{N},\qquad
D_{\text{общ}}=\frac{\sum_i(x_i-\bar x)^2n_i}{N}.$$


In [2]:
def mean_value(x, n):
    return sum(x * n) / sum(n)


def dispersion(x, n, mean):
    return sum((x - mean) ** 2 * n) / sum(n)


mean_age = mean_value(discrete["x"], discrete["n"])
D_total = dispersion(discrete["x"], discrete["n"], mean_age)

rows = []
for i in range(k):
    left = edges[i]
    right = edges[i + 1]
    if i < k - 1:
        group = discrete[(discrete["x"] >= left) & (discrete["x"] < right)]
    else:
        group = discrete[(discrete["x"] >= left) & (discrete["x"] <= right)]

    x = group["x"]
    n = group["n"]
    N_group = sum(n)
    mean_group = mean_value(x, n)
    D_group = dispersion(x, n, mean_group)
    rows.append([left, right, N_group, mean_group, D_group])

groups = pd.DataFrame(rows, columns=["x_l", "x_r", "N_j", "mean_j", "D_j"])
groups["D_within_part"] = groups["D_j"] * groups["N_j"] / N
groups["D_between_part"] = (groups["mean_j"] - mean_age) ** 2 * groups["N_j"] / N

display(groups.round(4))
print(f"Общая средняя возраста = {mean_age:.4f} года")
print(f"Общая дисперсия возраста = {D_total:.4f} года²")

,x_l,x_r,N_j,mean_j,D_j,D_within_part,D_between_part
0,14,23,4811,19.6134,3.3188,0.4925,36.8531
1,23,32,9476,27.0698,6.3823,1.8653,20.1497
2,32,41,7243,35.9502,6.3594,1.4206,0.0744
3,41,50,7951,45.0884,7.1235,1.7469,23.1468
4,50,59,1140,54.0640,9.3757,0.3297,12.2834
5,59,68,1472,62.8220,5.9914,0.2720,34.2064
6,68,77,330,69.8000,1.2933,0.0132,12.0631


Общая средняя возраста = 35.3730 года
Общая дисперсия возраста = 144.9170 года²


#### Внутригрупповая и межгрупповая дисперсии. Проверка формулы

$$D_{\text{внгр}}=\frac{\sum_{j=1}^k D_j N_j}{N},\qquad
D_{\text{межгр}}=\frac{\sum_{j=1}^k(\bar x_j-\bar x)^2N_j}{N}.$$

Формула общей дисперсии:

$$D_{\text{общ}}=D_{\text{внгр}}+D_{\text{межгр}}.$$

Внутригрупповая дисперсия описывает разброс возрастов внутри интервалов, межгрупповая — различия средних возрастов интервалов.

In [3]:
D_within = sum(groups["D_within_part"])
D_between = sum(groups["D_between_part"])
D_sum = D_within + D_between
error = abs(D_total - D_sum)

print(f"Внутригрупповая дисперсия = {D_within:.4f} года²")
print(f"Межгрупповая дисперсия = {D_between:.4f} года²")
print(f"Сумма дисперсий = {D_sum:.4f} года²")
print(f"Общая дисперсия = {D_total:.4f} года²")
print(f"Разность по модулю = {error:.10f}")

if error < 1e-10:
    print("Формула общей дисперсии выполняется с учётом погрешности вычислений.")
else:
    print("Формула общей дисперсии не выполняется.")

Внутригрупповая дисперсия = 6.1401 года²
Межгрупповая дисперсия = 138.7769 года²
Сумма дисперсий = 144.9170 года²
Общая дисперсия = 144.9170 года²
Разность по модулю = 0.0000000000
Формула общей дисперсии выполняется с учётом погрешности вычислений.


### 2. Корреляционное отношение

$$\eta=\sqrt{\frac{D_{\text{межгр}}}{D_{\text{общ}}}},\qquad 0\le\eta\le1.$$

$\eta^2$ — доля общей дисперсии возраста, приходящаяся на различия групповых средних. Корреляционное отношение не показывает направление связи.

В данном расчёте $\eta$ характеризует связь возраста с принадлежностью к возрастной группе. Связь возраста с абсолютной частотой оценивается далее коэффициентом Спирмена.

In [4]:
eta = (D_between / D_total) ** 0.5
eta_squared = eta ** 2

print(f"Корреляционное отношение η = {eta:.4f}")
print(f"Доля межгрупповой дисперсии η² = {eta_squared:.4f}")
print(f"Различия групповых средних составляют {eta_squared * 100:.2f}% дисперсии возраста.")
print("η характеризует связь возраста с его возрастной группой; направления связи не показывает.")

Корреляционное отношение η = 0.9786
Доля межгрупповой дисперсии η² = 0.9576
Различия групповых средних составляют 95.76% дисперсии возраста.
η характеризует связь возраста с его возрастной группой; направления связи не показывает.


### 3. Коэффициент ранговой корреляции Спирмена

$X$ — возраст, $Y$ — его абсолютная частота. Одна строка дискретного ряда соответствует одной паре. $m$ — число пар, $N$ — число людей.

Ранги присваиваются по возрастанию. Одинаковым значениям соответствует средний ранг.

$$R_0=1-\frac{6\sum_{i=1}^{m}d_i^2}{m(m^2-1)},\qquad d_i=R_{x_i}-R_{y_i}.$$

Сокращённая формула точна при отсутствии повторов. Частоты 26 и 188 повторяются, поэтому коэффициент уточняется через корреляцию средних рангов:

$$R=\frac{\sum_i(R_{x_i}-\bar R_x)(R_{y_i}-\bar R_y)}
{\sqrt{\sum_i(R_{x_i}-\bar R_x)^2\sum_i(R_{y_i}-\bar R_y)^2}}.$$


In [5]:
ranks = discrete.rename(columns={"n": "y"}).copy()
ranks["rank_x"] = ranks["x"].rank(method="average")
ranks["rank_y"] = ranks["y"].rank(method="average")
ranks["d"] = ranks["rank_x"] - ranks["rank_y"]
ranks["d_squared"] = ranks["d"] ** 2

m = len(ranks)
sum_d_squared = sum(ranks["d_squared"])
R_simple = 1 - 6 * sum_d_squared / (m * (m ** 2 - 1))

rank_x = ranks["rank_x"]
rank_y = ranks["rank_y"]
mean_rank_x = sum(rank_x) / m
mean_rank_y = sum(rank_y) / m
numerator = sum((rank_x - mean_rank_x) * (rank_y - mean_rank_y))
denominator = (sum((rank_x - mean_rank_x) ** 2) * sum((rank_y - mean_rank_y) ** 2)) ** 0.5
R = numerator / denominator

display(ranks)
print("Количество пар m =", m)
print(f"Сумма квадратов разностей рангов = {sum_d_squared:.4f}")
print(f"По сокращённой формуле R₀ = {R_simple:.6f}")
print(f"С учётом повторяющихся частот R = {R:.6f}")

if R < 0:
    print("Ранговая связь обратная: большему возрасту в целом соответствует меньшая частота.")
elif R > 0:
    print("Ранговая связь прямая: большему возрасту в целом соответствует большая частота.")
else:
    print("Ранговая корреляция равна нулю.")

,x,y,rank_x,rank_y,d,d_squared
0,14,49,1.0,5.0,-4.0,16.00
1,15,53,2.0,6.0,-4.0,16.00
2,16,236,3.0,24.0,-21.0,441.00
3,17,126,4.0,12.0,-8.0,64.00
4,18,806,5.0,37.0,-32.0,1024.00
5,19,971,6.0,45.0,-39.0,1521.00
6,20,910,7.0,43.0,-36.0,1296.00
7,21,716,8.0,32.0,-24.0,576.00
8,22,944,9.0,44.0,-35.0,1225.00
9,23,1071,10.0,53.0,-43.0,1849.00


Количество пар m = 58
Сумма квадратов разностей рангов = 49585.0000
По сокращённой формуле R₀ = -0.525270
С учётом повторяющихся частот R = -0.525317
Ранговая связь обратная: большему возрасту в целом соответствует меньшая частота.
